In [2]:
# INFERENCE & KAGGLE SUBMISSION
!pip install dagshub mlflow lightgbm scikit-learn cloudpickle -q

import pandas as pd
import dagshub
import mlflow
import warnings
warnings.filterwarnings('ignore')

# 1. Authenticate with DagsHub
REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"
dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)
mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

# 2. Load the Production Pipeline
model_name = "Walmart_LightGBM_Production_Pipeline"
model_version = 5  
model_uri = f"models:/{model_name}/{model_version}"

print(f"Loading {model_name} (Version {model_version}) from Model Registry...")
production_pipeline = mlflow.sklearn.load_model(model_uri)

# 3. Load the raw Kaggle Test Data
base = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'
raw_test = pd.read_csv(base + 'test.csv.zip', parse_dates=['Date'])
raw_features = pd.read_csv(base + 'features.csv.zip', parse_dates=['Date'])
raw_stores = pd.read_csv(base + 'stores.csv')

raw_test_data = raw_test.merge(raw_features, on=['Store', 'Date'], how='left', suffixes=('', '_feat')).merge(raw_stores, on='Store', how='left')
raw_X_test = raw_test_data.drop(columns=['IsHoliday_feat'])

# 4. Generate Predictions
# The transformer will securely handle the history and output the exact right order.
predictions = production_pipeline.predict(raw_X_test)

# 5. Format and Save Submission File
submission = pd.DataFrame({
    'Id': raw_test['Store'].astype(str) + '_' + raw_test['Dept'].astype(str) + '_' + raw_test['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': predictions
})

submission.to_csv('submission.csv', index=False)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 81.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 88.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0b2dce37-4a8e-48f3-b366-9214575abd91&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7d976e84a50da7b96bd47ccad7e1e743eb65323759abc70eeb11a452a2a180a1




Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

Loading Walmart_LightGBM_Production_Pipeline (Version 5) from Model Registry...


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20


In [3]:
from mlflow.tracking import MlflowClient

# Initialize the MLflow client
client = MlflowClient()

model_name = "Walmart_LightGBM_Production_Pipeline"
model_version = 5

client.set_registered_model_alias(
    name=model_name, 
    alias="best_model", 
    version=model_version
)